# MedRAG - 30 Negative-Control Questions

Runs ONLY the MedRAG arm on the 30 negative-control questions, checkpointing
each answer. Split out of notebook 11 so this heavy arm can be run and
debugged on its own.

Setup is identical to notebook 5 (MedCPT retriever, k=32, ollama_remote
generator, article-only T21_2026_09_02_ARTICLES corpus). Answers are written
to the SAME checkpoint file notebook 11 Cell 5 reads
(medrag_answers_negative_t21_2026_09_02_articles.json in the
negative_control_refusals output dir), so after this notebook finishes,
notebook 11 picks the answers up from cache and makes no MedRAG calls itself.

RUN WITH THE medrag KERNEL (the medrag conda env - Kernel > Change Kernel >
medrag), the same env the positive-control MedRAG benchmark (notebook 5) ran
in. Resume-safe: re-running the answer cell skips questions that already have
an answer.

## Cell 1 - Configuration

In [1]:
# ---- Cell 1: configuration ----
import json, os, re, sys, time
from pathlib import Path

import pandas as pd
import requests

NB_DIR   = Path.cwd()
ANALYSIS = (NB_DIR.parent if NB_DIR.name == 'nbs' else NB_DIR).resolve()
BENCH    = ANALYSIS / 'benchmarks'
OUT_DIR  = ANALYSIS / 'output' / 'ScientificReportRevisionV1' / 'negative_control_refusals'
OUT_DIR.mkdir(parents=True, exist_ok=True)

QUESTIONS_XLSX = ANALYSIS / 'input' / 'Revised_DownSyndrome_60_Questions_Final_linked.xlsx'
sys.path.insert(0, str(BENCH))

MEDRAG_CORPUS = 'T21_2026_09_02_ARTICLES'
MEDRAG_K      = 32
# Same checkpoint path notebook 11 Cell 5 uses - do not change one without the other
ANS_MEDRAG = OUT_DIR / f'medrag_answers_negative_{MEDRAG_CORPUS.lower()}.json'
print('questions  :', QUESTIONS_XLSX.name, '| exists:', QUESTIONS_XLSX.exists())
print('checkpoint :', ANS_MEDRAG)
print('corpus     :', MEDRAG_CORPUS, '| k =', MEDRAG_K)


questions  : Revised_DownSyndrome_60_Questions_Final_linked.xlsx | exists: True
checkpoint : /path/to/project/T21_research_assistant_v7/analysis/output/ScientificReportRevisionV1/negative_control_refusals/medrag_answers_negative_t21_2026_09_02_articles.json
corpus     : T21_2026_09_02_ARTICLES | k = 32


## Cell 2 - The 30 negative-control questions

In [2]:
# ---- Cell 2: the 30 negative-control questions ----
df_q = pd.read_excel(QUESTIONS_XLSX, sheet_name='Questions_for_testing')
df_q.columns = [c.strip() for c in df_q.columns]
neg = df_q[df_q['CTRL_Type'].str.strip().str.lower() == 'negative'].copy().reset_index(drop=True)
neg = neg.rename(columns={'Original question': 'question'})
neg['question'] = neg['question'].astype(str).str.strip()
neg['sl_no'] = neg['SLNo'].astype(str).str.strip().str.replace(r'^[Qq]', '', regex=True)
assert len(neg) == 30, f'expected 30 negative controls, found {len(neg)}'
print(f'negative controls: {len(neg)}')
neg[['sl_no', 'question']]


negative controls: 30


,sl_no,question
0,31,How do the PAM50 subtypes in Down syndrome aff...
1,32,What are some drugs that I can use to treat Hy...
2,33,How do pine trees relate to the development of...
3,34,Do people with Down syndrome have super human ...
4,35,How does the gene FX84A9 drive co-occurring co...
5,36,How does chloroplast dysfunction affect co-occ...
6,37,Why do all people with Down syndrome have the ...
7,38,Can Ivermectin be used to treat Down syndrome?
8,39,Are people with Down syndrome more prone to An...
9,40,What behavioral assays are used to study Down ...


## Cell 3 - Build MedRAG

Fails fast with a clear message if an import is missing - the LAST line of the
traceback names the module to install.

In [3]:
# ---- Cell 3: build MedRAG ----
from medrag_adapter import build_medrag, answer_freetext

t0 = time.time()
medrag = build_medrag(corpus=MEDRAG_CORPUS, retriever='MedCPT', provider='ollama_remote')
print(f'MedRAG ready in {time.time()-t0:.0f}s')
print('  generator :', medrag.model)
print('  retriever : MedCPT (ncbi/MedCPT-Query-Encoder)')
print(f'  corpus    : {MEDRAG_CORPUS} (flat ~1000-char chunks)')


/home/user/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/user/anaconda3/envs/medrag/lib/python3.10/site-packages/torch/cuda/__init__.py:228: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12060). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /__w/pytorch/pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0
/home/user/anaconda3/envs/medrag/lib/python3.10/site-packages/huggingface_hub/utils/_deprecation.py:131: FutureWarning: 'cached_download' (from 'huggingface_hub.file_downl

No sentence-transformers model found with name /home/user/.cache/torch/sentence_transformers/ncbi_MedCPT-Query-Encoder. Creating a new one with CLS pooling.
MedRAG ready in 3s
  generator : nemotron-3-super
  retriever : MedCPT (ncbi/MedCPT-Query-Encoder)
  corpus    : T21_2026_09_02_ARTICLES (flat ~1000-char chunks)


## Cell 4 - Run the 30 questions (checkpointed, resume-safe)

In [4]:
# ---- Cell 4: MedRAG answers ----
answers = json.loads(ANS_MEDRAG.read_text()) if ANS_MEDRAG.exists() else {}

for i, row in neg.iterrows():
    key, q = row['sl_no'], row['question']
    if answers.get(key, {}).get('answer'):
        print(f'  [{i+1}/30] [RESUME] SLNo {key}')
        continue
    t0 = time.time()
    try:
        text, snips = answer_freetext(medrag, q, k=MEDRAG_K)
        answers[key] = {'question': q, 'answer': text,
                        'secs': round(time.time() - t0, 1),
                        'snippet_ids': [s.get('id', '') for s in snips]}
    except Exception as exc:
        answers[key] = {'question': q, 'answer': '', 'error': str(exc)[:200]}
        print(f'  [{key}] ERROR {str(exc)[:110]}')
    ANS_MEDRAG.write_text(json.dumps(answers, indent=1, ensure_ascii=False))
    print(f'  [{i+1}/30] {answers[key].get("secs", "-")}s  {q[:62]}')

neg['medrag_answer'] = [answers.get(s, {}).get('answer', '') for s in neg['sl_no']]
n_ok = int((neg['medrag_answer'].str.len() > 0).sum())
print(f'\nanswered: {n_ok}/30')
if n_ok:
    print(f'mean answer length: {neg.loc[neg["medrag_answer"].str.len() > 0, "medrag_answer"].str.split().str.len().mean():.0f} words')
if n_ok < 30:
    print('re-run this cell to retry the failed/missing questions')


  [1/30] 15.7s  How do the PAM50 subtypes in Down syndrome affect treatment op
  [2/30] 29.1s  What are some drugs that I can use to treat Hypothyroidism in 
  [3/30] 15.4s  How do pine trees relate to the development of Down syndrome?
  [4/30] 11.5s  Do people with Down syndrome have super human strength?
  [5/30] 15.4s  How does the gene FX84A9 drive co-occurring conditions in Down
  [6/30] 14.5s  How does chloroplast dysfunction affect co-occurring condition
  [7/30] 38.7s  Why do all people with Down syndrome have the same diseases an
  [8/30] 21.1s  Can Ivermectin be used to treat Down syndrome?
  [9/30] 7.8s  Are people with Down syndrome more prone to Anthracnose?
  [10/30] 13.8s  What behavioral assays are used to study Down syndrome in yeas
  [11/30] 15.2s  Which gene-editing therapy has been approved to remove the ext
  [12/30] 24.2s  How does the chromosome 21 gene ZQKR7B contribute to interfero
  [13/30] 16.8s  What is the FDA-approved dose of tofacitinib for Down syndrome


---

Output: medrag_answers_negative_t21_2026_09_02_articles.json in
analysis/output/ScientificReportRevisionV1/negative_control_refusals/.
When all 30 are answered, run notebook 11 - its Cell 5 loads these answers
from the checkpoint and the classification/Excel/figure cells do the rest.